# ExperimentIQ — Day 6, Task 1: Document Chunker
================================================
Loads corpus documents from a folder, splits them into overlapping chunks,
and outputs a structured list ready for embedding.

WHY THIS DESIGN:
- Fixed-size chunking with overlap is the standard for RAG — overlap ensures
  no sentence is split across two chunks and loses its context.
- chunk_size=400 tokens ≈ 2-3 paragraphs. Enough context for the LLM to
  reason from, small enough to stay focused on one concept.
- chunk_overlap=80 tokens ≈ 20% overlap. The sweet spot — you capture
  boundary context without duplicating too much content.
- We tag every chunk with its source doc, chunk index, and char offsets.
  These metadata fields feed ChromaDB and let you trace every LLM claim
  back to its source — critical for debugging bad retrievals.

USAGE:
    python chunker.py                   # runs on ./corpus/ folder
    python chunker.py --test            # runs built-in tests only
    python chunker.py --folder my_docs  # custom folder

HOW TO TEST (see bottom of file or run --test flag):
    1. Unit tests validate chunk size, overlap, and metadata fields
    2. Inspection tests let you eyeball a real chunk visually
    3. Edge case tests check empty docs, tiny docs, unicode

In [1]:
import os
import re
import json
import argparse
from dataclasses import dataclass, asdict
from typing import List, Optional


In [27]:
CHUNK_SIZE = 400        # characters in each chunk (400 characters grabs about 50 - 60 words. 
CHUNK_OVERLAP = 80      # characters of overlap between consecutive chunks 
MIN_CHUNK_SIZE = 50     # discard chunks shorter than this (usually trailing whitespace)
CORPUS_FOLDER = "../documents"

In [33]:
# test_doc
with open('../documents/communicating_results_to_stakeholders.md', 'r') as file:
    content = file.read()

In [28]:
@dataclass
class Chunk:
    """
    Blueprint for a single chunk of text extracted from a corpus document.
    Contains the chunk content and metadata tracking its source, position, and size."""
    chunk_id: str           # unique ID: "{doc_name}__chunk_{index:04d}"
    source_doc: str         # filename of origin document
    chunk_index: int        # position of this chunk within its document
    text: str               # the actual chunk content
    char_start: int         # start character offset in the original document
    char_end: int           # end character offset in the original document
    word_count: int         # rough size signal for QA
    total_chunks_in_doc: int = 0  # filled in after all chunks are created


In [29]:
def clean_text(text: str) -> str:
    """
    Removes noise and trailing whitespace before chunking. 
    - Collapses 3+ newlines to 2 lines 
    """
    text = re.sub(r'\n{3,}', '\n\n', text) # collapse lines
    lines = [line.rstrip() for line in text.splitlines()] # removes trailing chars in a line
    return '\n'.join(lines).strip()

In [35]:
not content

False

In [34]:
if not content or len(content.strip()) < MIN_CHUNK_SIZE:
    print('too small')

In [30]:
def split_into_chunks(text: str, doc_name: str,
                      chunk_size: int = CHUNK_SIZE,
                      overlap: int = CHUNK_OVERLAP) -> List[Chunk]:

    if not text or len(text.strip()) < MIN_CHUNK_SIZE:
        return []

    # Sentence boundary markers, in priority order
    SPLIT_MARKERS = ['\n\n', '. ', '? ', '! ', '\n', '; ']

    chunks: List[Chunk] = []
    start = 0
    doc_len = len(text)

    while start < doc_len:
        end = min(start + chunk_size, doc_len)

        # If we're not at the end of the document, try to find a clean break
        if end < doc_len:
            best_break = end  # fallback: hard cut

            for marker in SPLIT_MARKERS:
                # Search backward from `end` for a sentence boundary
                search_start = max(start + MIN_CHUNK_SIZE, end - overlap)
                pos = text.rfind(marker, search_start, end)
                if pos != -1:
                    best_break = pos + len(marker)
                    break

            end = best_break

        raw_chunk = text[start:end]
        cleaned_chunk = raw_chunk.strip()

        if len(cleaned_chunk) >= MIN_CHUNK_SIZE:
            chunk_index = len(chunks)
            chunks.append(Chunk(
                chunk_id=f"{doc_name}__chunk_{chunk_index:04d}",
                source_doc=doc_name,
                chunk_index=chunk_index,
                text=cleaned_chunk,
                char_start=start,
                char_end=end,
                word_count=len(cleaned_chunk.split()),
            ))

        # Anchor next chunk relative to where this chunk ended, minus overlap
        next_start = end
        while next_start < doc_len and text[next_start] not in (' ', '\n'):
            next_start += 1
        start = min(next_start + 1, doc_len)

    # Back-fill total_chunks_in_doc now that we know the count
    for chunk in chunks:
        chunk.total_chunks_in_doc = len(chunks)

    return chunks
